# Recomendador documental Top-5

Isaac Esquivel Ruiz · Sistemas y tecnologías de administración del conocimiento

Caso académico relacionado con Softtek. Los 30 documentos son ficticios. Esta actividad es independiente de las tareas de ciclo de vida y ontología. Se utiliza la ruta A: TF-IDF y similitud coseno.

**Uso:** ejecutar todas las celdas en orden. El notebook incluye los datos y el código; no depende de un repo ni de Drive. La grabación de la reflexión y su enlace siguen pendientes, conforme a la consigna.

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

paquetes = {'numpy': 'numpy==2.3.5', 'pandas': 'pandas==2.2.3',
            'sklearn': 'scikit-learn==1.8.0', 'matplotlib': 'matplotlib==3.10.8'}
faltantes = [paquete for modulo, paquete in paquetes.items()
             if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltantes])
print('Dependencias disponibles. Python:', sys.version.split()[0])

Dependencias disponibles. Python: 3.13.18


## 1. Preparar el corpus y el módulo

La siguiente celda materializa la copia de demostración en `recomendador_softtek_demo`. Al volver a ejecutarla restaura esos dos archivos con su contenido original; conserva modificaciones personales fuera de esa carpeta.

In [2]:
BASE = Path.cwd() / 'recomendador_softtek_demo'
BASE.mkdir(exist_ok=True)
CSV_DEMO = 'id,title,text,category\nDOC001,Política de acceso y autenticación de cuentas,Toda cuenta de usuario debe tener acceso autorizado según su función. El acceso al portal requiere contraseña personal y autenticación multifactor. La mesa de servicio verifica la identidad antes de desbloquear una cuenta o restablecer credenciales. Los permisos deben revisarse cuando cambia el puesto del usuario. Las cuentas compartidas no están permitidas.,Políticas y procesos\nDOC002,Ticket: cuenta bloqueada en el portal,"Un usuario no pudo acceder al portal después de varios intentos con una contraseña incorrecta. Soporte verificó su identidad, revisó el bloqueo de la cuenta y restableció el acceso. Se confirmó la autenticación multifactor y se pidió al usuario actualizar las credenciales guardadas. El incidente quedó cerrado después de validar el inicio de sesión.",Soporte e incidentes\nDOC003,Proceso de alta y baja de permisos,"El responsable solicita el alta de la cuenta indicando los permisos requeridos. La mesa de servicio valida la autorización y habilita el acceso mínimo. Cuando termina la relación laboral, se desactiva la cuenta y se revocan las sesiones. Cada cambio de permisos debe quedar registrado para que la revisión de acceso identifique privilegios innecesarios.",Políticas y procesos\nDOC004,FAQ: recuperación de contraseña y segundo factor,"Si el usuario olvida su contraseña, debe solicitar la recuperación de acceso por el canal autorizado. Soporte verifica la identidad antes de restablecer la cuenta. Si pierde el segundo factor de autenticación, se revoca el dispositivo anterior y se registra uno nuevo. Nunca se solicita enviar contraseñas por correo para recuperar el acceso.",Operación y guías\nDOC005,Ticket: permisos insuficientes después de un cambio de puesto,"La cuenta de una analista inició sesión correctamente, pero no pudo abrir una sección del portal. La revisión encontró permisos del puesto anterior. Soporte recibió la autorización del responsable y ajustó el acceso según la nueva función. No fue necesario cambiar la contraseña porque la autenticación funcionaba. Se validó el acceso con la usuaria.",Soporte e incidentes\nDOC006,Revisión mensual de cuentas privilegiadas,"La revisión de acceso compara las cuentas privilegiadas con las autorizaciones vigentes. El equipo identifica cuentas inactivas, permisos excesivos y autenticación multifactor deshabilitada. Cada hallazgo se asigna a un responsable. Antes de retirar un permiso se verifica su uso operativo. La revisión conserva evidencia de las cuentas corregidas y de las excepciones autorizadas.",Operación y guías\nDOC007,Política de respaldos y restauración de datos,Los respaldos de datos deben ejecutarse según el calendario autorizado. El equipo verifica el resultado de cada copia y conserva la retención definida para el servicio. Un respaldo exitoso no demuestra que pueda restaurarse. Por eso se programan pruebas de restauración y se revisa la integridad de los archivos recuperados antes de considerar protegida la información.,Políticas y procesos\nDOC008,Ticket: respaldo nocturno fallido por falta de espacio,"El respaldo nocturno de la base de datos falló porque el repositorio de copias no tenía espacio disponible. Soporte revisó la retención, liberó copias vencidas autorizadas y repitió el respaldo. Después ejecutó una restauración de prueba y verificó la integridad de los datos. Se configuró una alerta de capacidad para evitar que el incidente se repita.",Soporte e incidentes\nDOC009,Proceso de restauración de un respaldo,Para restaurar datos se identifica la copia de respaldo correspondiente al punto de recuperación aprobado. La restauración se realiza primero en un entorno aislado. Se valida la integridad de archivos y registros antes de sustituir información operativa. El responsable registra el tiempo de recuperación y cualquier pérdida observada para ajustar el plan de respaldos.,Políticas y procesos\nDOC010,FAQ: diferencia entre respaldo completo e incremental,"El respaldo completo copia todos los datos seleccionados. El respaldo incremental guarda los cambios desde la copia anterior y requiere la cadena correspondiente para restaurar. Si falta una copia, la recuperación puede quedar incompleta. Antes de modificar la retención, el equipo debe comprobar que conserva los respaldos necesarios para reconstruir los datos.",Operación y guías\nDOC011,Ticket: restauración incompleta por copia dañada,"Durante una restauración de prueba se detectaron archivos dañados en una copia de respaldo. El equipo revisó la integridad, seleccionó una copia anterior y repitió la recuperación. Los datos recuperados se compararon con el inventario esperado. Se abrió una revisión del almacenamiento y se suspendió la eliminación de respaldos hasta asegurar una cadena de recuperación válida.",Soporte e incidentes\nDOC012,Bitácora de pruebas de recuperación,"La bitácora registra las pruebas de restauración de respaldos, la fecha de cada copia y el responsable de la recuperación. Incluye la integridad de los datos recuperados y el tiempo empleado. Cuando una prueba supera el objetivo de recuperación, se revisan el tamaño del respaldo, la capacidad del repositorio y las instrucciones de restauración.",Operación y guías\nDOC013,Política de despliegue y reversión de versiones,"El despliegue de una nueva versión requiere pruebas aprobadas y una ventana de cambio. Antes de publicar se conserva la versión anterior y se documenta el procedimiento de reversión. Si aparecen errores críticos, el responsable detiene el despliegue y ejecuta el rollback autorizado. La decisión debe considerar el impacto en el servicio y los cambios de datos.",Políticas y procesos\nDOC014,Ticket: errores después del despliegue de una versión,"Después del despliegue de una nueva versión, el portal comenzó a devolver errores. El equipo comparó los registros con la versión anterior y detectó una configuración incompatible. Se ejecutó la reversión mediante rollback y se verificó la recuperación del servicio. El despliegue quedó suspendido hasta repetir las pruebas de compatibilidad y obtener una nueva autorización.",Soporte e incidentes\nDOC015,Proceso de autorización de cambios de software,"La solicitud de cambio identifica la versión, el objetivo del despliegue y el impacto esperado. El responsable revisa las pruebas, la ventana de mantenimiento y el procedimiento de reversión. No se autoriza una publicación sin evidencia de validación. Si el cambio modifica datos, el plan explica cómo tratar esa modificación durante un posible rollback.",Políticas y procesos\nDOC016,Guía de rollback ante una publicación defectuosa,Para realizar el rollback se detiene la publicación y se identifica la última versión estable. La reversión restaura la configuración compatible y requiere revisar los cambios de datos. Después se ejecutan pruebas de disponibilidad y se confirman los registros de error. El despliegue solo se reanuda cuando la causa de la falla ha sido corregida.,Operación y guías\nDOC017,Ticket: despliegue detenido por pruebas de integración,Las pruebas de integración detectaron errores antes de publicar la nueva versión. El equipo detuvo el despliegue y conservó la versión estable en producción. No fue necesario ejecutar rollback porque la publicación no había ocurrido. Se corrigió la incompatibilidad y se solicitaron nuevas pruebas antes de aprobar otra ventana de cambio.,Soporte e incidentes\nDOC018,Lista de verificación de una liberación,"Antes del despliegue, la lista verifica la versión aprobada, las pruebas terminadas y la disponibilidad del responsable. También confirma que existe un procedimiento de reversión y una versión anterior recuperable. Después de la publicación se observan errores y tiempos de respuesta. Si la validación falla, se evalúa el rollback según el impacto del cambio.",Operación y guías\nDOC019,Política de atención de correos sospechosos,"Los mensajes que soliciten datos sensibles o contengan enlaces inesperados deben reportarse como posible phishing. El usuario no debe abrir archivos sospechosos ni responder al remitente. Seguridad conserva las cabeceras del correo para analizar el origen. Si hubo interacción con el enlace, se revisan el equipo y las credenciales expuestas antes de cerrar la alerta.",Políticas y procesos\nDOC020,Ticket: correo de phishing con enlace externo,Una persona recibió un correo que simulaba una notificación urgente y solicitaba abrir un enlace externo. Reportó el mensaje sin interactuar. Seguridad analizó el remitente y confirmó el intento de phishing. El enlace fue bloqueado y el mensaje se retiró de otros buzones. Se compartieron indicadores del correo para reconocer intentos similares.,Soporte e incidentes\nDOC021,Proceso de aislamiento de equipos comprometidos,"Cuando se detecta malware, el equipo se aísla de la red según el procedimiento autorizado. Seguridad conserva evidencias y analiza archivos sospechosos antes de la limpieza. El responsable revisa si el incidente comenzó con un correo de phishing. La reconexión requiere verificar que se eliminó la amenaza y que no persisten indicadores de compromiso.",Políticas y procesos\nDOC022,FAQ: señales de un mensaje fraudulento,"Un correo de phishing puede usar urgencia, un remitente parecido al legítimo o enlaces que llevan a un dominio diferente. Los archivos inesperados también requieren revisión. El usuario debe reportar el mensaje sospechoso y conservarlo para análisis. No todos los errores de redacción prueban fraude; seguridad debe evaluar el contexto y el destino del enlace.",Operación y guías\nDOC023,Ticket: archivo malicioso abierto desde un correo,"El usuario abrió un archivo adjunto de un correo sospechoso y se generó una alerta de malware. Seguridad aisló el equipo, conservó las evidencias y analizó el archivo. La investigación identificó una campaña de phishing. Se revisaron otros buzones y se bloquearon los indicadores detectados antes de autorizar la reconexión del dispositivo.",Soporte e incidentes\nDOC024,Bitácora de simulación de phishing,"La simulación de phishing registra cuántos usuarios reportaron el correo sospechoso y cuántos abrieron el enlace de prueba. El objetivo es identificar necesidades de capacitación, sin publicar resultados individuales. El equipo revisa los mensajes utilizados y explica señales de riesgo. Las métricas se comparan entre ejercicios de alcance similar para evitar conclusiones engañosas.",Operación y guías\nDOC025,Política de monitoreo de rendimiento,"El monitoreo de rendimiento observa latencia, uso de CPU y consumo de memoria de los servicios. Las alertas se definen según la carga habitual y el impacto para los usuarios. Una alerta de CPU no demuestra por sí sola una falla. El equipo debe correlacionar métricas y registros antes de reiniciar un servicio o ampliar capacidad.",Políticas y procesos\nDOC026,Ticket: latencia elevada por saturación de CPU,"El servicio mostró latencia elevada durante una carga de trabajo extraordinaria. El monitoreo detectó saturación de CPU y aumento del tiempo de respuesta. El equipo identificó un proceso intensivo, ajustó la concurrencia y verificó las métricas posteriores. No se atribuyó el problema a memoria porque su consumo permaneció dentro del comportamiento habitual.",Soporte e incidentes\nDOC027,Proceso de análisis de capacidad,"El análisis de capacidad compara la carga prevista con las métricas de CPU, memoria y latencia. Se revisan tendencias de uso y periodos de saturación antes de ampliar recursos. El responsable define una prueba de carga para verificar el cambio. La decisión debe equilibrar rendimiento, costo y margen operativo, sin basarse en un único pico de consumo.",Políticas y procesos\nDOC028,Guía de diagnóstico de lentitud del servicio,"Cuando un servicio presenta lentitud se revisan la latencia, la CPU, la memoria y las consultas a la base de datos. El diagnóstico compara métricas durante periodos normales y durante la falla. Si el problema coincide con mayor carga, se prueba una reducción controlada de concurrencia. Toda intervención debe validar el tiempo de respuesta posterior.",Operación y guías\nDOC029,Ticket: consumo creciente de memoria,El monitoreo registró consumo creciente de memoria y aumento de latencia después de varias horas de operación. La CPU se mantuvo estable. El equipo analizó el proceso y encontró objetos que no se liberaban. Se aplicó una corrección y se repitió la prueba de carga para confirmar que el uso de memoria dejó de crecer.,Soporte e incidentes\nDOC030,Resumen de métricas de disponibilidad y latencia,"El resumen operativo presenta latencia, disponibilidad, carga y uso de recursos por servicio. Las métricas de CPU y memoria se muestran junto con el volumen de solicitudes para explicar variaciones. Un promedio puede ocultar periodos de saturación, por lo que también se revisan percentiles. El informe sirve para priorizar investigaciones, no para atribuir causas automáticamente.",Operación y guías\n'
CODIGO = '"""Recomendación Top-K mediante TF-IDF y similitud coseno.\n\nCaso académico con documentos ficticios inspirados en servicios de TI.\nNo se utilizan category ni id como variables de similitud.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport re\nimport sys\nimport textwrap\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nSTOPWORDS = frozenset(\'a al algo antes como con cuando de del desde donde durante e el ella en entre es esta este estos fue ha hacia hasta la las le lo los mas mediante muy o para pero por porque que se segun ser si sin sobre su sus un una uno unos y\'.split())\n\n\ndef limpiar_texto(texto: str) -> str:\n    """Minúsculas, normalización de tildes y eliminación de símbolos.\n\n    Se conserva la palabra \'no\' para no borrar todas las negaciones, aunque\n    TF-IDF no interpreta su sentido de forma fiable.\n    """\n    normal = unicodedata.normalize(\'NFKD\', str(texto).lower())\n    normal = \'\'.join(c for c in normal if not unicodedata.combining(c))\n    normal = re.sub(r\'[^a-z0-9\\s]\', \' \', normal)\n    return \' \'.join(t for t in normal.split() if t not in STOPWORDS)\n\n\nclass Recomendador:\n    """Índice TF-IDF ajustado una sola vez al corpus completo."""\n\n    def __init__(self, corpus: str | Path = \'corpus.csv\') -> None:\n        self.ruta = Path(corpus)\n        self.df = pd.read_csv(self.ruta, dtype=str, keep_default_na=False)\n        required = {\'id\', \'title\', \'text\'}\n        if not required.issubset(self.df.columns):\n            raise ValueError(\'El CSV necesita las columnas id, title y text.\')\n        if \'category\' not in self.df:\n            self.df[\'category\'] = \'Sin categoría\'\n        if len(self.df) < 2:\n            raise ValueError(\'Se requieren al menos dos documentos.\')\n        for col in required:\n            self.df[col] = self.df[col].str.strip()\n            if self.df[col].eq(\'\').any():\n                raise ValueError(f\'Hay valores vacíos en {col}.\')\n        if self.df[\'id\'].duplicated().any():\n            raise ValueError(\'Los identificadores deben ser únicos.\')\n        textos = self.df[\'title\'] + \' \' + self.df[\'text\']\n        limpios = textos.map(limpiar_texto)\n        if limpios.eq(\'\').any():\n            raise ValueError(\'Hay documentos sin contenido después de la limpieza.\')\n        if limpios.duplicated().any():\n            raise ValueError(\'Hay documentos duplicados después de la limpieza.\')\n        self.vectorizador = TfidfVectorizer(\n            preprocessor=limpiar_texto,\n            lowercase=False,\n            ngram_range=(1, 2),\n            sublinear_tf=True,\n            norm=\'l2\',\n            min_df=1,\n        )\n        self.matriz = self.vectorizador.fit_transform(textos)\n        if np.any(np.asarray(self.matriz.getnnz(axis=1)) == 0):\n            raise ValueError(\'Algún documento no genera términos válidos.\')\n        # Pequeño corpus: se calcula explícitamente la matriz N x N solicitada.\n        self.similitudes = np.clip(cosine_similarity(self.matriz), 0.0, 1.0)\n        self.indices = {doc_id: i for i, doc_id in enumerate(self.df[\'id\'])}\n\n    def recomendar(self, *, documento_id: str | None = None,\n                   consulta: str | None = None, k: int = 5) -> pd.DataFrame:\n        """Devuelve hasta K coincidencias positivas, ordenadas por score e ID.\n\n        El score mide semejanza léxica, no probabilidad de que la solución sirva.\n        Una consulta fuera del vocabulario devuelve una tabla vacía.\n        """\n        if (documento_id is None) == (consulta is None):\n            raise ValueError(\'Proporciona un ID o una consulta, pero no ambos.\')\n        if isinstance(k, bool) or not isinstance(k, int) or k < 1:\n            raise ValueError(\'K debe ser un entero positivo.\')\n        semilla = None\n        if documento_id is not None:\n            documento_id = documento_id.strip()\n            if documento_id not in self.indices:\n                raise ValueError(f\'ID desconocido: {documento_id}\')\n            semilla = self.indices[documento_id]\n            puntuaciones = self.similitudes[semilla].copy()\n            puntuaciones[semilla] = -1.0\n        else:\n            if not consulta or not limpiar_texto(consulta):\n                raise ValueError(\'La consulta no puede estar vacía.\')\n            vector = self.vectorizador.transform([consulta])\n            puntuaciones = np.clip(cosine_similarity(vector, self.matriz).ravel(), 0.0, 1.0)\n        resultado = self.df[[\'id\', \'title\', \'category\']].copy()\n        resultado[\'similarity\'] = puntuaciones\n        resultado = resultado.loc[resultado[\'similarity\'] > 0]\n        resultado = resultado.sort_values([\'similarity\', \'id\'], ascending=[False, True]).head(k)\n        resultado.insert(0, \'rank\', range(1, len(resultado) + 1))\n        return resultado.reset_index(drop=True)\n\n\n# Juicios de relevancia temática establecidos por el diseño del corpus.\n# No se derivan de las puntuaciones ni se utilizan para ajustar el vectorizador.\nPRUEBAS = [\n    dict(nombre=\'prueba_1_politica\', entrada=\'DOC001\', tipo=\'id\',\n         relevantes=[\'DOC002\', \'DOC003\', \'DOC004\', \'DOC005\', \'DOC006\'],\n         criterio=\'Documentos de acceso, cuentas, permisos y autenticación.\'),\n    dict(nombre=\'prueba_2_incidente\', entrada=\'DOC008\', tipo=\'id\',\n         relevantes=[\'DOC007\', \'DOC009\', \'DOC010\', \'DOC011\', \'DOC012\'],\n         criterio=\'Documentos de respaldo, retención, integridad y restauración.\'),\n    dict(nombre=\'prueba_3_consulta\',\n         entrada=\'El despliegue de una nueva versión produjo errores. Necesito el procedimiento de reversión o rollback y las pruebas para recuperar el servicio.\',\n         tipo=\'texto\', relevantes=[\'DOC013\', \'DOC014\', \'DOC015\', \'DOC016\', \'DOC017\', \'DOC018\'],\n         criterio=\'Documentos de despliegue, validación de versiones y reversión.\'),\n]\n\n\ndef comentario_prueba(prueba: dict, resultados: pd.DataFrame) -> list[str]:\n    aciertos = int(resultados[\'id\'].isin(prueba[\'relevantes\']).sum())\n    top = str(resultados.iloc[0][\'title\']) if len(resultados) else \'Sin coincidencias\'\n    if prueba[\'nombre\'].startswith(\'prueba_1\'):\n        contenido = [\n            \'Usé una política de acceso como punto de partida para buscar información relacionada.\',\n            f\'La primera recomendación fue: {top}.\',\n            f\'{aciertos} de las cinco recomendaciones pertenecen al tema de acceso y autenticación.\',\n            \'La lista permite pasar de una política general a guías y casos que ayudan a aplicarla.\',\n            \'El documento inicial se excluye para que no ocupe un lugar entre las recomendaciones.\',\n            \'La similitud orienta la búsqueda, pero el técnico debe revisar los permisos y el contexto.\',\n        ]\n    elif prueba[\'nombre\'].startswith(\'prueba_2\'):\n        contenido = [\n            \'Partí de un incidente donde falló el respaldo por falta de espacio.\',\n            f\'La primera recomendación fue: {top}.\',\n            f\'{aciertos} de las cinco recomendaciones tratan respaldos y recuperación de datos.\',\n            \'DOC002 trata una cuenta bloqueada y es un resultado ajeno al problema de respaldos.\',\n            \'El vocabulario genérico compartido puede acercar incidentes que tienen causas distintas.\',\n            \'Revisaría el contenido y añadiría filtros temáticos antes de usar una recomendación.\',\n        ]\n    else:\n        contenido = [\n            \'En esta prueba escribí el problema sin conocer el identificador de ningún documento.\',\n            f\'La primera recomendación fue: {top}.\',\n            f\'{aciertos} de las cinco recomendaciones corresponden al tema de despliegue y reversión.\',\n            \'La consulta muestra que se pueden recuperar guías usando una descripción breve del problema.\',\n            \'El resultado depende de palabras compartidas como versión, despliegue, pruebas y rollback.\',\n            \'Si el usuario utiliza otros términos, podría ser necesario reformular la consulta.\',\n        ]\n    return contenido\n\n\ndef generar_evidencias(modelo: Recomendador, carpeta: str | Path = \'evidencias\') -> list[dict]:\n    """Ejecuta las tres pruebas y exporta resultados, criterios e imágenes."""\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n\n    carpeta = Path(carpeta)\n    carpeta.mkdir(parents=True, exist_ok=True)\n    resumen = []\n    for prueba in PRUEBAS:\n        kwargs = {\'documento_id\': prueba[\'entrada\']} if prueba[\'tipo\'] == \'id\' else {\'consulta\': prueba[\'entrada\']}\n        resultados = modelo.recomendar(**kwargs, k=5)\n        if len(resultados) != 5:\n            raise AssertionError(f"{prueba[\'nombre\']}: se esperaban cinco resultados.")\n        if prueba[\'tipo\'] == \'id\':\n            assert prueba[\'entrada\'] not in resultados[\'id\'].tolist()\n        assert resultados[\'similarity\'].is_monotonic_decreasing\n        assert resultados[\'id\'].is_unique\n        assert resultados[\'similarity\'].between(0, 1).all()\n        aciertos = int(resultados[\'id\'].isin(prueba[\'relevantes\']).sum())\n        precision = aciertos / 5\n        comentario = comentario_prueba(prueba, resultados)\n        texto = (\'PRUEBA: \' + prueba[\'nombre\'] + \'\\nEntrada: \' + prueba[\'entrada\']\n                 + \'\\nCriterio de relevancia: \' + prueba[\'criterio\']\n                 + \'\\n\\n\' + resultados.to_string(index=False, float_format=lambda x: f\'{x:.6f}\')\n                 + f\'\\n\\nP@5 temática: {precision:.2f} ({aciertos}/5)\\n\'\n                 + \'\\n\'.join(comentario) + \'\\n\')\n        (carpeta / (prueba[\'nombre\'] + \'.txt\')).write_text(texto, encoding=\'utf-8\')\n        fig = plt.figure(figsize=(13, 8), facecolor=\'white\')\n        fig.text(.045, .94, prueba[\'nombre\'].replace(\'_\', \' \').capitalize(), size=19, weight=\'bold\')\n        fig.text(.045, .895, textwrap.fill(\'Entrada: \' + prueba[\'entrada\'], 115), va=\'top\', size=10)\n        ax = fig.add_axes([.045, .39, .91, .38]); ax.axis(\'off\')\n        rows = [[str(row[\'rank\']), row[\'id\'], textwrap.fill(row[\'title\'], 46), f"{row[\'similarity\']:.4f}"] for row in resultados.to_dict(\'records\')]\n        tabla = ax.table(cellText=rows, colLabels=[\'Top\', \'ID\', \'Título\', \'Similitud coseno\'],\n                         colWidths=[.06, .11, .65, .18], cellLoc=\'left\', bbox=[0, 0, 1, 1])\n        tabla.auto_set_font_size(False); tabla.set_fontsize(10)\n        for (r, c), cell in tabla.get_celld().items():\n            cell.set_edgecolor(\'black\'); cell.set_linewidth(.6)\n            cell.set_facecolor(\'white\')\n            if r == 0: cell.set_text_props(weight=\'bold\')\n        fig.text(.045, .35, f\'P@5 temática: {precision:.2f} ({aciertos}/5). Evaluación ilustrativa en corpus simulado.\', size=11, weight=\'bold\')\n        fig.text(.045, .305, \'\\n\'.join(textwrap.fill(x, 132) for x in comentario), size=10, va=\'top\', linespacing=1.5)\n        fig.text(.045, .045, \'Salida real del script. La similitud no representa una probabilidad de éxito.\', size=9)\n        fig.savefig(carpeta / (prueba[\'nombre\'] + \'.png\'), dpi=160, bbox_inches=\'tight\')\n        plt.close(fig)\n        resumen.append({**prueba, \'precision_at_5\': precision,\n                        \'resultados\': resultados.to_dict(\'records\'), \'comentario\': comentario})\n        print(texto)\n    # Registra versión, corpus y parámetros para relacionar evidencia y ejecución.\n    import sklearn\n    registro = {\n        \'corpus_sha256\': hashlib.sha256(modelo.ruta.read_bytes()).hexdigest(),\n        \'documentos\': len(modelo.df), \'terminos\': modelo.matriz.shape[1],\n        \'versiones\': {\'python\': sys.version.split()[0], \'scikit_learn\': sklearn.__version__, \'pandas\': pd.__version__, \'numpy\': np.__version__},\n        \'parametros\': {\'ngram_range\': [1, 2], \'sublinear_tf\': True, \'norm\': \'l2\', \'campos\': [\'title\', \'text\'], \'k\': 5},\n        \'evaluacion\': \'Relevancia temática manual en corpus sintético. No evalúa resolución efectiva ni generalización.\',\n        \'pruebas\': resumen,\n    }\n    (carpeta / \'resultados.json\').write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    return resumen\n\n\ndef validar_comportamiento(modelo: Recomendador) -> list[str]:\n    """Controles de salida y de entradas que podrían producir recomendaciones engañosas."""\n    controles = []\n    assert np.allclose(modelo.similitudes, modelo.similitudes.T)\n    assert np.allclose(np.diag(modelo.similitudes), 1)\n    controles.append(\'Matriz coseno simétrica y diagonal igual a 1: OK.\')\n    salida = modelo.recomendar(documento_id=\'DOC001\')\n    assert len(salida) == 5 and \'DOC001\' not in salida[\'id\'].tolist()\n    assert salida[\'similarity\'].is_monotonic_decreasing and salida[\'id\'].is_unique\n    controles.append(\'Top-5 ordenado, sin duplicados ni documento semilla: OK.\')\n    assert salida.equals(modelo.recomendar(documento_id=\'DOC001\'))\n    controles.append(\'Resultado repetible con el mismo corpus: OK.\')\n    for entrada in [dict(documento_id=\'INEXISTENTE\'), dict(consulta=\'   \'),\n                    dict(documento_id=\'DOC001\', consulta=\'acceso\'),\n                    dict(documento_id=\'DOC001\', k=0)]:\n        try:\n            modelo.recomendar(**entrada)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(f\'Entrada inválida aceptada: {entrada}\')\n    controles.append(\'ID inválido, consulta vacía, dos modalidades y K=0 rechazados: OK.\')\n    assert modelo.recomendar(consulta=\'zzzxxyyqq\').empty\n    controles.append(\'Consulta fuera del vocabulario: tabla vacía, sin resultados inventados: OK.\')\n    return controles\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--corpus\', default=\'corpus.csv\')\n    selector = parser.add_mutually_exclusive_group()\n    selector.add_argument(\'--id\', dest=\'documento_id\')\n    selector.add_argument(\'--texto\')\n    selector.add_argument(\'--pruebas\', action=\'store_true\')\n    parser.add_argument(\'--k\', type=int, default=5)\n    parser.add_argument(\'--salida\', default=\'evidencias\')\n    args = parser.parse_args()\n    try:\n        modelo = Recomendador(args.corpus)\n        if args.pruebas:\n            generar_evidencias(modelo, args.salida)\n            controles = validar_comportamiento(modelo)\n            Path(args.salida, \'validacion.txt\').write_text(\'\\n\'.join(controles) + \'\\n\', encoding=\'utf-8\')\n            print(\'\\n\'.join(controles))\n            return 0\n        if args.documento_id is None and args.texto is None:\n            modo = input(\'Modo: 1 = ID, 2 = texto: \').strip()\n            if modo == \'1\': args.documento_id = input(\'ID (por ejemplo DOC001): \').strip()\n            elif modo == \'2\': args.texto = input(\'Describe el problema: \').strip()\n            else: raise ValueError(\'Selecciona 1 o 2.\')\n        tabla = modelo.recomendar(documento_id=args.documento_id, consulta=args.texto, k=args.k)\n        print(\'Entrada:\', args.documento_id if args.documento_id is not None else args.texto)\n        if tabla.empty:\n            print(\'Sin coincidencias léxicas. Reformula con términos del dominio.\')\n        else:\n            print(tabla.to_string(index=False, float_format=lambda x: f\'{x:.6f}\'))\n            if len(tabla) < args.k:\n                print(f\'Solo hay {len(tabla)} documentos con similitud positiva.\')\n        return 0\n    except (ValueError, OSError, pd.errors.ParserError, EOFError) as exc:\n        print(f\'Error: {exc}\', file=sys.stderr)\n        return 1\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'
(BASE / 'corpus.csv').write_text(CSV_DEMO, encoding='utf-8')
(BASE / 'recomendador.py').write_text(CODIGO, encoding='utf-8')
import importlib.util
especificacion = importlib.util.spec_from_file_location('recomendador_demo', BASE / 'recomendador.py')
modulo = importlib.util.module_from_spec(especificacion)
especificacion.loader.exec_module(modulo)
from IPython.display import display, Image
modelo = modulo.Recomendador(BASE / 'corpus.csv')
print('Documentos:', len(modelo.df))
print('Columnas:', ', '.join(modelo.df.columns))
display(modelo.df.groupby('category').size().rename('documentos').reset_index())
display(modelo.df[['id', 'title', 'category']].head())

Documentos: 30
Columnas: id, title, text, category


,category,documentos
0,Operación y guías,10
1,Políticas y procesos,10
2,Soporte e incidentes,10


,id,title,category
0,DOC001,Política de acceso y autenticación de cuentas,Políticas y procesos
1,DOC002,Ticket: cuenta bloqueada en el portal,Soporte e incidentes
2,DOC003,Proceso de alta y baja de permisos,Políticas y procesos
3,DOC004,FAQ: recuperación de contraseña y segundo factor,Operación y guías
4,DOC005,Ticket: permisos insuficientes después de un c...,Soporte e incidentes


**Interpretación del corpus**

Preparé treinta documentos para representar situaciones comunes de un equipo de TI.  
Hay políticas, incidentes y guías, con diez documentos en cada categoría.  
Los temas incluyen accesos, respaldos, despliegues, seguridad y rendimiento.  
Los datos son simulados y no describen documentos internos reales de Softtek.  
La categoría sirve para revisar el conjunto, pero no se usa para calcular la similitud.  
Esta variedad permite buscar relaciones entre una política y ejemplos de cómo aplicarla.

## 2. Representación TF-IDF

El módulo limpia el texto, une título y contenido y genera palabras y pares de palabras. TF-IDF destaca términos según su frecuencia en cada documento y en el corpus. La similitud coseno compara la orientación de los vectores. En este caso produce puntuaciones de 0 a 1, sin interpretarlas como probabilidades.

In [3]:
print('Matriz TF-IDF:', modelo.matriz.shape)
print('Matriz de similitudes:', modelo.similitudes.shape)
print('Ejemplo de limpieza:', modulo.limpiar_texto('¡Revisión de ACCESO y autenticación!'))
print('Número de características:', len(modelo.vectorizador.get_feature_names_out()))

Matriz TF-IDF: (30, 1488)
Matriz de similitudes: (30, 30)
Ejemplo de limpieza: revision acceso autenticacion
Número de características: 1488


**Interpretación de la representación**

Cada fila de la matriz representa un documento y cada columna un término o par de términos.  
La limpieza permite comparar palabras aunque cambien las mayúsculas o las tildes.  
Los términos frecuentes en casi todos los documentos suelen aportar menos para diferenciarlos.  
El modelo conserva información de palabras compartidas, pero no comprende todo su significado.  
La matriz de similitudes permite comparar cada documento con los otros veintinueve.  
Al recomendar se quita la semilla para evitar que el sistema la devuelva como mejor resultado.

## Prueba 1: política/proceso

In [4]:
prueba = modulo.PRUEBAS[0]
print('Entrada:', prueba['entrada'])
print('Criterio:', prueba['criterio'])
r1 = modelo.recomendar(documento_id=prueba['entrada'], k=5)
display(r1.round({'similarity': 6}))
p1 = r1['id'].isin(prueba['relevantes']).sum() / 5
print('P@5 temática:', round(p1, 2))

Entrada: DOC001
Criterio: Documentos de acceso, cuentas, permisos y autenticación.


,rank,id,title,category,similarity
0,1,DOC002,Ticket: cuenta bloqueada en el portal,Soporte e incidentes,0.140721
1,2,DOC004,FAQ: recuperación de contraseña y segundo factor,Operación y guías,0.134736
2,3,DOC005,Ticket: permisos insuficientes después de un c...,Soporte e incidentes,0.124003
3,4,DOC006,Revisión mensual de cuentas privilegiadas,Operación y guías,0.114726
4,5,DOC003,Proceso de alta y baja de permisos,Políticas y procesos,0.100431


P@5 temática: 1.0


**Interpretación**

Usé una política de acceso como punto de partida para buscar información relacionada.  
La primera recomendación fue: Ticket: cuenta bloqueada en el portal.  
5 de las cinco recomendaciones pertenecen al tema de acceso y autenticación.  
La lista permite pasar de una política general a guías y casos que ayudan a aplicarla.  
El documento inicial se excluye para que no ocupe un lugar entre las recomendaciones.  
La similitud orienta la búsqueda, pero el técnico debe revisar los permisos y el contexto.

## Prueba 2: soporte/incidente

In [5]:
prueba = modulo.PRUEBAS[1]
print('Entrada:', prueba['entrada'])
print('Criterio:', prueba['criterio'])
r2 = modelo.recomendar(documento_id=prueba['entrada'], k=5)
display(r2.round({'similarity': 6}))
p2 = r2['id'].isin(prueba['relevantes']).sum() / 5
print('P@5 temática:', round(p2, 2))

Entrada: DOC008
Criterio: Documentos de respaldo, retención, integridad y restauración.


,rank,id,title,category,similarity
0,1,DOC012,Bitácora de pruebas de recuperación,Operación y guías,0.090867
1,2,DOC011,Ticket: restauración incompleta por copia dañada,Soporte e incidentes,0.087552
2,3,DOC010,FAQ: diferencia entre respaldo completo e incr...,Operación y guías,0.062210
3,4,DOC007,Política de respaldos y restauración de datos,Políticas y procesos,0.061663
4,5,DOC002,Ticket: cuenta bloqueada en el portal,Soporte e incidentes,0.055285


P@5 temática: 0.8


**Interpretación**

Partí de un incidente donde falló el respaldo por falta de espacio.  
La primera recomendación fue: Bitácora de pruebas de recuperación.  
4 de las cinco recomendaciones tratan respaldos y recuperación de datos.  
DOC002 trata una cuenta bloqueada y es un resultado ajeno al problema de respaldos.  
El vocabulario genérico compartido puede acercar incidentes que tienen causas distintas.  
Revisaría el contenido y añadiría filtros temáticos antes de usar una recomendación.

## Prueba 3: consulta de texto libre

In [6]:
prueba = modulo.PRUEBAS[2]
print('Entrada:', prueba['entrada'])
print('Criterio:', prueba['criterio'])
r3 = modelo.recomendar(consulta=prueba['entrada'], k=5)
display(r3.round({'similarity': 6}))
p3 = r3['id'].isin(prueba['relevantes']).sum() / 5
print('P@5 temática:', round(p3, 2))

Entrada: El despliegue de una nueva versión produjo errores. Necesito el procedimiento de reversión o rollback y las pruebas para recuperar el servicio.
Criterio: Documentos de despliegue, validación de versiones y reversión.


,rank,id,title,category,similarity
0,1,DOC013,Política de despliegue y reversión de versiones,Políticas y procesos,0.350469
1,2,DOC014,Ticket: errores después del despliegue de una ...,Soporte e incidentes,0.349828
2,3,DOC018,Lista de verificación de una liberación,Operación y guías,0.200293
3,4,DOC017,Ticket: despliegue detenido por pruebas de int...,Soporte e incidentes,0.195535
4,5,DOC015,Proceso de autorización de cambios de software,Políticas y procesos,0.160176


P@5 temática: 1.0


**Interpretación**

En esta prueba escribí el problema sin conocer el identificador de ningún documento.  
La primera recomendación fue: Política de despliegue y reversión de versiones.  
5 de las cinco recomendaciones corresponden al tema de despliegue y reversión.  
La consulta muestra que se pueden recuperar guías usando una descripción breve del problema.  
El resultado depende de palabras compartidas como versión, despliegue, pruebas y rollback.  
Si el usuario utiliza otros términos, podría ser necesario reformular la consulta.

## 3. Evidencias y controles

Los conjuntos de relevancia están declarados en el módulo antes de las pruebas y se basan en el tema de cada documento. P@5 = documentos temáticamente relevantes en las cinco recomendaciones / 5. Es una evaluación ilustrativa, no una validación de soluciones efectivas en producción.

In [7]:
import contextlib
import io
captura = io.StringIO()
with contextlib.redirect_stdout(captura):
    pruebas = modulo.generar_evidencias(modelo, BASE / 'evidencias')
controles = modulo.validar_comportamiento(modelo)
(BASE / 'evidencias' / 'validacion.txt').write_text('\n'.join(controles) + '\n', encoding='utf-8')
for control in controles:
    print(control)
print('P@5 media:', round((p1 + p2 + p3) / 3, 4))
print('Evidencias generadas en:', BASE / 'evidencias')

Matriz coseno simétrica y diagonal igual a 1: OK.
Top-5 ordenado, sin duplicados ni documento semilla: OK.
Resultado repetible con el mismo corpus: OK.
ID inválido, consulta vacía, dos modalidades y K=0 rechazados: OK.
Consulta fuera del vocabulario: tabla vacía, sin resultados inventados: OK.
P@5 media: 0.9333
Evidencias generadas en: /content/recomendador_softtek_demo/evidencias


**Interpretación de la validación**

La precisión temática media fue de 0.9333 en las tres pruebas de demostración.  
La segunda prueba muestra un falso positivo porque incluyó un ticket de cuenta bloqueada.  
El vocabulario genérico puede acercar dos documentos que no resuelven el mismo problema.  
También comprobé que no se repita la semilla y que las entradas inválidas se rechacen.  
Una consulta fuera del vocabulario devuelve una lista vacía en vez de resultados arbitrarios.  
Estos controles verifican el funcionamiento, pero aún falta evaluar la utilidad con usuarios reales.

## 4. Probar otra entrada

Cambia `modo` por `id` o `texto` y escribe tu entrada. Esta celda no modifica las tres evidencias anteriores.

In [8]:
modo = 'texto' # @param ['id', 'texto']
entrada = 'correo sospechoso con enlace de phishing' # @param {type:'string'}
try:
    if modo == 'id':
        recomendaciones = modelo.recomendar(documento_id=entrada)
    elif modo == 'texto':
        recomendaciones = modelo.recomendar(consulta=entrada)
    else:
        raise ValueError('El modo debe ser id o texto.')
    print('Entrada:', entrada)
    if recomendaciones.empty:
        print('Sin coincidencias léxicas; intenta reformular la consulta.')
    else:
        display(recomendaciones.round({'similarity': 6}))
except ValueError as error:
    print(error)

Entrada: correo sospechoso con enlace de phishing


,rank,id,title,category,similarity
0,1,DOC024,Bitácora de simulación de phishing,Operación y guías,0.226310
1,2,DOC020,Ticket: correo de phishing con enlace externo,Soporte e incidentes,0.199915
2,3,DOC023,Ticket: archivo malicioso abierto desde un correo,Soporte e incidentes,0.194855
3,4,DOC022,FAQ: señales de un mensaje fraudulento,Operación y guías,0.146401
4,5,DOC019,Política de atención de correos sospechosos,Políticas y procesos,0.102156


## Reflexión organizacional

Las recomendaciones pueden acercar experiencias previas y evitar que cada técnico investigue desde cero.  
También ayudan a localizar procedimientos antes de decidir una acción, sin sustituir el juicio del especialista.  
TF-IDF tiene límites con sinónimos, ambigüedad, negaciones, documentos cortos y vocabulario poco representativo.  
Para un uso real agregaría versión, vigencia, responsable, sistema afectado y permisos de acceso por rol.  
Compararía filtros temáticos y embeddings con más consultas, usando relevancia y utilidad revisadas por técnicos.  
El ahorro de tiempo y la mejora en decisiones deben medirse; todavía no se demostraron en esta actividad.

**Video requerido:** grabar una reflexión de máximo cinco minutos usando `Guion_reflexion.md` como apoyo. El enlace real debe agregarse al reporte antes de entregar. Estas notas no sustituyen la grabación.

Documentación: [TF-IDF](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) · [Similitud coseno](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html).